In [11]:
!pip install gudhi

In [12]:
import os
import time
import warnings
from typing import Dict, List, Tuple, Set

warnings.filterwarnings("ignore")
import gudhi
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
import pandas as pd


INPUT_CSV = '/content/DNA-repair-edges.csv'      # CSV de entrada (arestas)
GENE_COL_1 = 'Source'
GENE_COL_2 = 'Target'

# --- Genes que devem aparecer destacados
GENES_TO_HIGHLIGHT: Set[str] = {}

# --- Dimensão máxima do complexo da filtração ---
MAXIMAL_DIMENSION = 3

# --- Quantos genes remover (um de cada vez) para o teste de robustez ---
N_GENES_TO_REMOVE = None
GENERATE_PLOTS_FOR_REMOVALS = True
OINTS = 200
TRUNCATION = True

TCentrality = Dict[int, float]
TSimplexSet = Set[Tuple]
TSimplex_float_field = Dict[Tuple, float]
TPersistence = List[Tuple[int, Tuple[float, float]]]

# =============================================================================
# LEITURA DO CSV E CONSTRUÇÃO DA REDE
# =============================================================================

def Load_Network_From_Csv(FileName: str, Col_1: str, Col_2: str) -> nx.Graph:
    Df_Edges = pd.read_csv(FileName)
    Df_Edges = Df_Edges[[Col_1, Col_2]].dropna()

    Graph = nx.Graph()
    Global_Dict: Dict[str, int] = {}

    for _, Row in Df_Edges.iterrows():
        Source_Name = str(Row[Col_1]).strip()
        Target_Name = str(Row[Col_2]).strip()

        if not Source_Name or not Target_Name or Source_Name == Target_Name:
            continue

        if Source_Name not in Global_Dict:
            Global_Dict[Source_Name] = len(Global_Dict)
        if Target_Name not in Global_Dict:
            Global_Dict[Target_Name] = len(Global_Dict)

        Source_Id = Global_Dict[Source_Name]
        Target_Id = Global_Dict[Target_Name]

        if not Graph.has_node(Source_Id):
            Graph.add_node(Source_Id, Name=Source_Name)
        if not Graph.has_node(Target_Id):
            Graph.add_node(Target_Id, Name=Target_Name)

        Graph.add_edge(min(Source_Id, Target_Id), max(Source_Id, Target_Id))

    Num_Components = nx.number_connected_components(Graph)
    if Num_Components > 1:
        Largest_Nodes = max(nx.connected_components(Graph), key=len)
        Graph = Graph.subgraph(Largest_Nodes).copy()
        print(f"A rede tinha {Num_Components} componentes; usando apenas a maior componente conexa "
              f"({Graph.number_of_nodes()} genes, {Graph.number_of_edges()} interações).")

    return Graph

# =============================================================================
# PLOT DA REDE COMPLETA
# =============================================================================

def Plot_Network(Graph: nx.Graph, Output_Path: str):
    Labels = {Node: Graph.nodes[Node]['Name'] for Node in Graph.nodes()}

    plt.figure(figsize=(16, 16), dpi=200)
    Layout = nx.kamada_kawai_layout(Graph)
    nx.draw_networkx(
        Graph,
        Layout,
        labels=Labels,
        node_size=180,
        font_size=8,
        font_weight="bold",
        width=0.7,
        edge_color="lightgray",
        node_color="violet",
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(Output_Path, dpi=600, bbox_inches="tight")
    plt.close()
    print(f"Plot da rede completa salvo em: {Output_Path}")

# =============================================================================
# COMPLEXO DE CLIQUES
# =============================================================================

def Find_All_Simplices_Cliques(Graph: nx.Graph, Max_Dim: int) -> TSimplexSet:
    Simplex_Set: TSimplexSet = {(v,) for v in Graph.nodes()}
    if Max_Dim == 0:
        return Simplex_Set

    Arestas = {(u, v) if u < v else (v, u) for u, v in Graph.edges()}
    Simplex_Set.update(Arestas)
    if Max_Dim == 1 or not Arestas:
        return Simplex_Set

    Dic_Vizinhos = {v: {n for n in Graph.neighbors(v) if n > v} for v in Graph.nodes()}

    Triangulos = set()
    for u, v in Arestas:
        for w in Dic_Vizinhos[v].intersection(Dic_Vizinhos[u]):
            Triangulos.add((u, v, w))
    Simplex_Set.update(Triangulos)
    if Max_Dim == 2 or not Triangulos:
        return Simplex_Set

    Tetraedros = set()
    for u, v, w in Triangulos:
        for z in Dic_Vizinhos[w].intersection(Dic_Vizinhos[v], Dic_Vizinhos[u]):
            Tetraedros.add((u, v, w, z))
    Simplex_Set.update(Tetraedros)
    if Max_Dim == 3 or not Tetraedros:
        return Simplex_Set

    Pentacoros = set()
    for u, v, w, z in Tetraedros:
        for t in Dic_Vizinhos[z].intersection(Dic_Vizinhos[w], Dic_Vizinhos[v], Dic_Vizinhos[u]):
            Pentacoros.add((u, v, w, z, t))
    Simplex_Set.update(Pentacoros)

    return Simplex_Set

# =============================================================================
# CENTRALIDADES (GRAU)
# =============================================================================

def Calculate_Centrality(Graph: nx.Graph, Round_Values: bool = False) -> TCentrality:
    Centrality = {Node: float(Degree) for Node, Degree in Graph.degree()}

    if Round_Values:
        return {Node: round(Value, 10) for Node, Value in Centrality.items()}
    else:
        return Centrality

# =============================================================================
# FILTRAÇÃO VFB e HOMOLOGIA PERSISTENTE
# =============================================================================

def Evaluate_All_Simplices(Simplex_Set: TSimplexSet, Centrality: TCentrality, Ascending: bool) -> TSimplex_float_field:
    Factor = 1 if Ascending else -1
    return {Simplex: float(max(Factor * Centrality[v] for v in Simplex)) for Simplex in Simplex_Set}

def Compute_Persistence(Simplices: TSimplexSet, Centrality: TCentrality, Ascending: bool, Max_Dim: int):
    Simplex_Tree = gudhi.SimplexTree()
    Simplex_Float_Field = Evaluate_All_Simplices(Simplices, Centrality, Ascending)
    for Simplex, Value in Simplex_Float_Field.items():
        Simplex_Tree.insert(list(Simplex), filtration=Value)
    Persistence = Simplex_Tree.persistence()
    Betti = Simplex_Tree.betti_numbers()
    Betti = (Betti + [0] * (Max_Dim + 1))[:Max_Dim + 1]
    return Persistence, Betti

# =============================================================================
# DISTÂNCIA DE BOTTLENECK ENTRE DOIS DIAGRAMAS
# =============================================================================

def Format_Intervals_To_Matrix(Persistence_List: TPersistence, Dim: int, T_Max: float) -> np.ndarray:
    Intervals = []
    for Pt_Dim, (b, d) in Persistence_List:
        if Pt_Dim != Dim:
            continue
        if d == float('inf'):
            d = T_Max
        Intervals.append((b, d))
    return np.array(Intervals) if Intervals else np.empty((0, 2))

def Get_T_Max(*Persistences: TPersistence) -> float:
    T_Max = 0.0
    for Persistence in Persistences:
        for _, (b, d) in Persistence:
            T_Max = max(T_Max, d if d != float('inf') else b)
    return T_Max

def Calculate_Bottleneck_Per_Dimension(Global_Persistence: TPersistence, Node_Persistence: TPersistence, Max_Dim: int) -> Dict[int, float]:
    T_Max = Get_T_Max(Global_Persistence, Node_Persistence) if TRUNCATION else float('inf')
    Result = {}
    for Dim in range(Max_Dim + 1):
        Global_M = Format_Intervals_To_Matrix(Global_Persistence, Dim, T_Max)
        Node_M = Format_Intervals_To_Matrix(Node_Persistence, Dim, T_Max)
        if len(Global_M) == 0 and len(Node_M) == 0:
            Result[Dim] = 0.0
        else:
            Result[Dim] = gudhi.bottleneck_distance(Global_M, Node_M)
    return Result

# =============================================================================
# FUNÇÕES DE PLOT (DIAGRAMA, BARCODE)
# =============================================================================

def Save_Persistence_Plots(Persistence: TPersistence, Max_Dim: int, Output_Prefix: str, Title: str):
    # --- Diagrama de persistência ---
    Fig, Ax = plt.subplots(figsize=(6, 6))
    gudhi.plot_persistence_diagram(Persistence, axes=Ax)
    Ax.set_title(f"Diagrama de Persistência - {Title}")
    plt.savefig(f"{Output_Prefix}_diagrama.png", dpi=150, bbox_inches='tight')
    plt.close(Fig)

    # --- Barcode de persistência ---
    Dimensions_Len = {}
    Max_Dim_In_Barcode = max((Bd[0] for Bd in Persistence), default=0)
    for Dim in range(0, Max_Dim_In_Barcode + 1):
        Dimensions_Len[Dim] = len([Bd for Bd in Persistence if Bd[0] == Dim])

    Dimension_Resume = ''
    for Dim, Count in Dimensions_Len.items():
        Dimension_Resume += f"{Dim}: {Count} | "
    Dimension_Resume = Dimension_Resume[:-3]

    Fig, Ax = plt.subplots(figsize=(8, 6))
    gudhi.plot_persistence_barcode(Persistence, axes=Ax)
    Ax.set_title(f"Barcode - {Title}\n{Dimension_Resume}")
    plt.savefig(f"{Output_Prefix}_barcode.png", dpi=150, bbox_inches='tight')
    plt.close(Fig)

# =============================================================================
#  TABELA DE RESULTADO
# =============================================================================

def Export_Table(Rows: List[Dict], FileName: str, Title: str):
    Df = pd.DataFrame(Rows)
    with pd.ExcelWriter(FileName, engine='openpyxl') as Writer:
        Df.to_excel(Writer, sheet_name='Resultados', index=False, startrow=1)

    Wb = openpyxl.load_workbook(FileName)
    Ws = Wb['Resultados']
    Ws['A1'] = Title
    Ws['A1'].font = Font(bold=True, size=12)

    Header_Fill = PatternFill(start_color="000000", end_color="000000", fill_type="solid")
    Header_Font = Font(color="FFFFFF", bold=True)
    Center_Align = Alignment(horizontal="center", vertical="center")

    for Col in Ws.iter_cols(min_row=2, max_row=Ws.max_row):
        Header_Cell = Col[0]
        Header_Cell.fill = Header_Fill
        Header_Cell.font = Header_Font
        Max_Len = 0
        for Cell in Col:
            Cell.alignment = Center_Align
            if Cell.value is not None:
                Max_Len = max(Max_Len, len(str(Cell.value)))
        Ws.column_dimensions[Col[0].column_letter].width = Max_Len + 4

    Wb.save(FileName)

# =============================================================================
#  ANÁLISE
# =============================================================================
def Run_Analysis(
    Graph: nx.Graph,
    Simplices: TSimplexSet,
    Ascending: bool,
    Base_Output_Dir: str,
    Max_Dim: int,
    Genes_To_Remove: List[int]
):

    Order = 'Crescente' if Ascending else 'Decrescente'
    Combo_Name = f"Grau"
    Combo_Dir = os.path.join(Base_Output_Dir, f"{Combo_Name}_{Order}")
    Plots_Dir = os.path.join(Combo_Dir, 'Plots')
    os.makedirs(Plots_Dir, exist_ok=True)


    # --- (a) Rede completa ---
    Start_Time = time.perf_counter()
    Global_Centrality = Calculate_Centrality(Graph)
    Global_Persistence, Global_Betti = Compute_Persistence(Simplices, Global_Centrality, Ascending, Max_Dim)

    Global_Plots_Dir = os.path.join(Plots_Dir, 'Rede_Completa')
    os.makedirs(Global_Plots_Dir, exist_ok=True)
    Save_Persistence_Plots(Global_Persistence, Max_Dim, os.path.join(Global_Plots_Dir, 'Global'),
                            f"Rede completa (Grau, {Order})")

    Rows = [{
        'Gene_Removido': '(Nenhum - Rede Completa)',
        'Grau': '-',
        **{f'Betti_H{D}': Global_Betti[D] for D in range(Max_Dim + 1)},
        **{f'Bars_H{D}': len([P for P in Global_Persistence if P[0] == D]) for D in range(Max_Dim + 1)},
        **{f'Bottleneck_H{D}': 0.0 for D in range(Max_Dim + 1)},
    }]

    # --- (b) Remoção de genes, um a um ---
    Total = len(Genes_To_Remove)
    for i, Node in enumerate(Genes_To_Remove, 1):
        Gene_Name = Graph.nodes[Node]['Name']
        Gene_Degree = Global_Centrality[Node]
        print(f"  [ Removendo gene {i}/{Total}: {Gene_Name}")

        Subgraph = Graph.copy()
        Subgraph.remove_node(Node)

        Node_Centrality = Calculate_Centrality(Subgraph)
        Simplices_Sem_No = {S for S in Simplices if Node not in S}
        Node_Persistence, Node_Betti = Compute_Persistence(Simplices_Sem_No, Node_Centrality, Ascending, Max_Dim)

        Bottleneck = Calculate_Bottleneck_Per_Dimension(Global_Persistence, Node_Persistence, Max_Dim)

        Rows.append({
            'Gene_Removido': Gene_Name,
            'Grau': Gene_Degree,
            **{f'Betti_H{D}': Node_Betti[D] for D in range(Max_Dim + 1)},
            **{f'Bars_H{D}': len([P for P in Node_Persistence if P[0] == D]) for D in range(Max_Dim + 1)},
            **{f'Bottleneck_H{D}': Bottleneck[D] for D in range(Max_Dim + 1)},
        })

        if GENERATE_PLOTS_FOR_REMOVALS:
            Gene_Plots_Dir = os.path.join(Plots_Dir, f'Removido_{Gene_Name}')
            os.makedirs(Gene_Plots_Dir, exist_ok=True)
            Save_Persistence_Plots(Node_Persistence, Max_Dim, os.path.join(Gene_Plots_Dir, Gene_Name),
                                    f"Removendo {Gene_Name} (Grau, {Order})")


    def Get_Degree(Row):
        if Row['Gene_Removido'].startswith('(Nenhum'):
            return float('inf')
        return Row['Grau']
    Rows_Sorted = [Rows[0]] + sorted(Rows[1:], key=Get_Degree, reverse=True)

    Table_Path = os.path.join(Combo_Dir, f'Tabela_{Combo_Name}_{Order}.xlsx')
    Export_Table(Rows_Sorted, Table_Path, f"Centralidade: Grau  |  Ordem: {Order}")

    Duration = time.perf_counter() - Start_Time
    print(f"Combinação Grau/{Order} finalizada em {Duration:.2f}s. Tabela salva em: {Table_Path}")

# =============================================================================
# FUNÇÃO PRINCIPAL
# =============================================================================

def Main():
    Total_Start = time.perf_counter()

    Graph = Load_Network_From_Csv(INPUT_CSV, GENE_COL_1, GENE_COL_2)
    Plot_Network(Graph, os.path.join('Rede_Completa.png'))
    Simplices = Find_All_Simplices_Cliques(Graph, MAXIMAL_DIMENSION)

    Nodes_Sorted = sorted(Graph.nodes(), key=lambda N: Graph.nodes[N]['Name'])
    if N_GENES_TO_REMOVE is None:
        Genes_To_Remove = Nodes_Sorted
        print(f"Modo completo: todos os {len(Genes_To_Remove)} genes serão removidos, um a um.")
    else:
        Genes_To_Remove = Nodes_Sorted[:N_GENES_TO_REMOVE]
        print(f"Modo teste: apenas os primeiros {len(Genes_To_Remove)} genes serão removidos "
              f"(ajuste N_GENES_TO_REMOVE para mudar isso).")

    output_base_dir = 'Output_Analysis'
    Run_Analysis(
        Graph,
        Simplices,
        True,
        output_base_dir,
        MAXIMAL_DIMENSION,
        Genes_To_Remove
    )

    Total_Duration = time.perf_counter() - Total_Start
    print(f"\nTEMPO TOTAL DE EXECUÇÃO: {Total_Duration:.2f}s ({Total_Duration/60:.2f} min)")


if __name__ == "__main__":
    Main()


Plot da rede completa salvo em: Rede_Completa.png
Modo completo: todos os 132 genes serão removidos, um a um.
  [ Removendo gene 1/132: ABL1
  [ Removendo gene 2/132: APBB1
  [ Removendo gene 3/132: APEX1
  [ Removendo gene 4/132: ATM
  [ Removendo gene 5/132: ATR
  [ Removendo gene 6/132: BABAM1
  [ Removendo gene 7/132: BAP1
  [ Removendo gene 8/132: BARD1
  [ Removendo gene 9/132: BRCA1
  [ Removendo gene 10/132: BRCA2
  [ Removendo gene 11/132: BRCC3
  [ Removendo gene 12/132: BRIP1
  [ Removendo gene 13/132: CDK2
  [ Removendo gene 14/132: CETN2
  [ Removendo gene 15/132: CHD1L
  [ Removendo gene 16/132: CHEK1
  [ Removendo gene 17/132: CHEK2
  [ Removendo gene 18/132: CLSPN
  [ Removendo gene 19/132: DCAF8L1
  [ Removendo gene 20/132: DCLRE1A
  [ Removendo gene 21/132: DCLRE1C
  [ Removendo gene 22/132: DNA2
  [ Removendo gene 23/132: DTL
  [ Removendo gene 24/132: ELL
  [ Removendo gene 25/132: EME1
  [ Removendo gene 26/132: ERCC1
  [ Removendo gene 27/132: ERCC4
  [ Removendo 